# Day 07 — Backpropagation Deep Dive

> **Goal:** Understand the chain rule and manually trace backpropagation through a multi-layer network.

## The Chain Rule in Plain English

If `y` depends on `u`, and `u` depends on `x`, then:
$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

**Analogy:** If doubling the engine power (x→u) doubles the speed, and doubling the speed (u→y) doubles the fuel consumption, then doubling engine power quadruples fuel consumption.

In a neural network, each layer is a link in the chain. Backprop applies the chain rule from the loss backward through every layer.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

## 1. Chain Rule with a Simple Graph

In [ ]:
# f(x) = (3x + 2)²
# Let u = 3x + 2, then f = u²
# df/dx = df/du * du/dx = 2u * 3 = 6(3x + 2)

x = torch.tensor(1.0, requires_grad=True)
u = 3 * x + 2  # u = 5
f = u ** 2      # f = 25

f.backward()

print(f"x = {x.item()}")
print(f"u = 3x + 2 = {u.item()}")
print(f"f = u² = {f.item()}")
print(f"\ndf/dx (autograd) = {x.grad.item()}")
print(f"df/dx (manual)   = 6 * (3*{x.item()} + 2) = {6 * (3*x.item() + 2)}")

## 2. Manual Backprop Through a 2-Layer Network

Let's build a tiny 2-layer network and manually trace the gradients.

```
Input: x
Layer 1: z1 = w1 * x + b1
Activation: a1 = ReLU(z1)
Layer 2: z2 = w2 * a1 + b2
Loss: L = (z2 - y_true)²
```

In [ ]:
# ── Forward pass (manual) ──────────────────────────────
x = torch.tensor(2.0)
y_true = torch.tensor(1.0)

# Learnable parameters
w1 = torch.tensor(0.5, requires_grad=True)
b1 = torch.tensor(0.1, requires_grad=True)
w2 = torch.tensor(-0.3, requires_grad=True)
b2 = torch.tensor(0.2, requires_grad=True)

# Layer 1
z1 = w1 * x + b1
a1 = torch.relu(z1)  # ReLU activation

# Layer 2
z2 = w2 * a1 + b2

# Loss
loss = (z2 - y_true) ** 2

print("── Forward Pass ──")
print(f"x = {x.item():.4f}")
print(f"z1 = w1*x + b1 = {z1.item():.4f}")
print(f"a1 = ReLU(z1) = {a1.item():.4f}")
print(f"z2 = w2*a1 + b2 = {z2.item():.4f}")
print(f"loss = (z2 - y_true)² = {loss.item():.4f}")

### Manual Backward Pass — Applying the Chain Rule

Now comes the key insight: **backpropagation is just the chain rule applied systematically from the loss back to the parameters.** We compute gradients layer by layer, multiplying local gradients as we go.

Each gradient tells us: "If I increase this parameter slightly, how much does the loss change?"

In [ ]:
# ── Backward pass (manual computation) ─────────────────
# Let's compute gradients by hand using the chain rule

# dL/dz2 = 2 * (z2 - y_true)
dL_dz2 = 2 * (z2.item() - y_true.item())
print("── Manual Backward Pass ──")
print(f"dL/dz2 = {dL_dz2:.4f}")

# dL/dw2 = dL/dz2 * dz2/dw2 = dL/dz2 * a1
dL_dw2 = dL_dz2 * a1.item()
print(f"dL/dw2 = dL/dz2 * a1 = {dL_dw2:.4f}")

# dL/db2 = dL/dz2 * dz2/db2 = dL/dz2 * 1
dL_db2 = dL_dz2
print(f"dL/db2 = dL/dz2 = {dL_db2:.4f}")

# dL/da1 = dL/dz2 * dz2/da1 = dL/dz2 * w2
dL_da1 = dL_dz2 * w2.item()
print(f"dL/da1 = dL/dz2 * w2 = {dL_da1:.4f}")

# dL/dz1 = dL/da1 * da1/dz1 = dL/da1 * (1 if z1>0 else 0)  [ReLU derivative]
relu_grad = 1.0 if z1.item() > 0 else 0.0
dL_dz1 = dL_da1 * relu_grad
print(f"dL/dz1 = dL/da1 * ReLU'(z1) = {dL_dz1:.4f}")

# dL/dw1 = dL/dz1 * dz1/dw1 = dL/dz1 * x
dL_dw1 = dL_dz1 * x.item()
print(f"dL/dw1 = dL/dz1 * x = {dL_dw1:.4f}")

# dL/db1 = dL/dz1 * 1
dL_db1 = dL_dz1
print(f"dL/db1 = dL/dz1 = {dL_db1:.4f}")

### Verification: Manual vs Autograd

Let's now run `loss.backward()` to let PyTorch compute the same gradients automatically, and compare them to our manual calculations. They should match exactly — this confirms both that our chain rule derivation is correct and that autograd works as expected.

In [ ]:
# ── Verify with autograd ───────────────────────────────
loss.backward()

print("── Verify: Manual vs Autograd ──")
print(f"dL/dw1: manual={dL_dw1:.4f}, autograd={w1.grad.item():.4f}, match={abs(dL_dw1 - w1.grad.item()) < 1e-5}")
print(f"dL/db1: manual={dL_db1:.4f}, autograd={b1.grad.item():.4f}, match={abs(dL_db1 - b1.grad.item()) < 1e-5}")
print(f"dL/dw2: manual={dL_dw2:.4f}, autograd={w2.grad.item():.4f}, match={abs(dL_dw2 - w2.grad.item()) < 1e-5}")
print(f"dL/db2: manual={dL_db2:.4f}, autograd={b2.grad.item():.4f}, match={abs(dL_db2 - b2.grad.item()) < 1e-5}")
print("\n✅ All gradients match! Autograd does the chain rule for us.")

## 3. Visualizing ReLU and Its Gradient

In [ ]:
x_vals = torch.linspace(-3, 3, 200)
relu_vals = torch.relu(x_vals)
relu_grad_vals = (x_vals > 0).float()  # 1 if x>0, else 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_vals.numpy(), relu_vals.numpy(), linewidth=2, color="blue")
axes[0].set_title("ReLU: max(0, x)", fontsize=13)
axes[0].set_xlabel("x")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].axvline(0, color="black", linewidth=0.5)

axes[1].plot(x_vals.numpy(), relu_grad_vals.numpy(), linewidth=2, color="red")
axes[1].set_title("ReLU Gradient: 1 if x>0, else 0", fontsize=13)
axes[1].set_xlabel("x")
axes[1].set_ylim(-0.2, 1.5)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].axvline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

print("When ReLU's input is negative, the gradient is ZERO.")
print("This means those neurons don't learn — this is called the 'dying ReLU' problem.")

## 4. Backprop Through a Batch — Vectorized

In [ ]:
torch.manual_seed(42)

# Data: 4 samples, 2 features each
X = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0],
                  [7.0, 8.0]])
y_true = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

# Simple 2-layer network
W1 = torch.randn(2, 3, requires_grad=True)  # 2 inputs → 3 hidden
b1 = torch.zeros(3, requires_grad=True)
W2 = torch.randn(3, 1, requires_grad=True)  # 3 hidden → 1 output
b2 = torch.zeros(1, requires_grad=True)

# Forward pass (vectorized — all 4 samples at once)
z1 = X @ W1 + b1          # (4, 2) @ (2, 3) + (3,) → (4, 3)
a1 = torch.relu(z1)        # (4, 3)
z2 = a1 @ W2 + b2          # (4, 3) @ (3, 1) + (1,) → (4, 1)
loss = ((z2 - y_true) ** 2).mean()

print(f"Input shape:  {X.shape}")
print(f"Hidden shape: {a1.shape}")
print(f"Output shape: {z2.shape}")
print(f"Loss: {loss.item():.4f}")

# Backward — one call computes ALL gradients
loss.backward()

print(f"\nW1.grad shape: {W1.grad.shape}")
print(f"W2.grad shape: {W2.grad.shape}")
print("\n✅ Autograd handles batches and matrix operations automatically!")

---
## ✏️ Exercises

In [ ]:
# Exercise 1: Compute the gradient of f(x) = sigmoid(3x - 1) at x = 0.5
# sigmoid(z) = 1 / (1 + exp(-z)), sigmoid'(z) = sigmoid(z) * (1 - sigmoid(z))
# Use autograd AND verify by hand.

# Your code here:

In [ ]:
# Exercise 2: Add a third layer to the network in Section 4.
# W3 (1 hidden → 1 output). Run forward + backward and check all gradient shapes.

# Your code here:

---
## 📝 Key Takeaways

1. **Backpropagation** = chain rule applied backward through the computation graph.
2. Each node receives gradients from above and passes "blame" backward.
3. **ReLU gradient** is 0 for negative inputs (neurons can "die").
4. Autograd handles arbitrarily complex graphs — you just call `.backward()`.
5. Vectorized backprop through batches works the same — just with matrices.

**Tomorrow:** We'll learn `nn.Module` — PyTorch's way of packaging all this into clean, reusable models!